# Notebook 05: Recommendations
**Objective:** Synthesise findings into actionable recommendations for the deal desk — optimal discount thresholds by segment, approval SLAs, and rep coaching priorities.

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
deals_df = pd.read_csv('../data/raw/deals.csv')
approvals_df = pd.read_csv('../data/raw/approvals.csv')
outcomes_df = pd.read_csv('../data/raw/outcomes.csv')

deals_df['created_date'] = pd.to_datetime(deals_df['created_date'])
approvals_df['submitted_date'] = pd.to_datetime(approvals_df['submitted_date'])
approvals_df['decision_date'] = pd.to_datetime(approvals_df['decision_date'])
outcomes_df['close_date'] = pd.to_datetime(outcomes_df['close_date'])

con = duckdb.connect()
con.register('deals', deals_df)
con.register('approvals', approvals_df)
con.register('outcomes', outcomes_df)

merged = deals_df.merge(outcomes_df, on='deal_id')

print(f"deals:     {len(deals_df):,} rows")
print(f"approvals: {len(approvals_df):,} rows")
print(f"outcomes:  {len(outcomes_df):,} rows")

In [ ]:
bins = [0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.50]
labels = ['0-5%', '5-10%', '10-15%', '15-20%', '20-25%', '25%+']

nb_only = merged[merged['deal_type'] == 'New Business'].copy()
nb_only['discount_band'] = pd.cut(
    nb_only['discount_pct'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

threshold_analysis = (
    nb_only.groupby(['segment', 'discount_band'], observed=True)
    .agg(
        deals=('deal_id', 'count'),
        win_rate=('win_flag', 'mean')
    )
    .reset_index()
)

## Recommendation 1: Optimal Discount Thresholds by Segment

In [ ]:
for seg in ['SMB', 'Mid-Market', 'Enterprise']:
    d = threshold_analysis[
        (threshold_analysis['segment'] == seg) &
        (threshold_analysis['deals'] >= 20)
    ]
    peak_idx = d['win_rate'].idxmax()
    print(f"\n--- {seg} ---")
    print(f"{'Band':<12} {'Deals':>8} {'Win Rate':>10}")
    print("-" * 32)
    for idx, row in d.iterrows():
        marker = ' ← peak' if idx == peak_idx else ''
        print(
            f"{str(row['discount_band']):<12} {row['deals']:>8,} "
            f"{row['win_rate']:>9.1%}{marker}"
        )

## Recommended Discount Thresholds

In [ ]:
thresholds = {
    'SMB':        ('10-15%', '15%'),
    'Mid-Market': ('15-20%', '20%'),
    'Enterprise': ('5-10%',  '10%')
}

print(
    f"{'Segment':<15} {'Peak Win Rate Band':>20} "
    f"{'Recommended Cap':>18} {'Current Avg':>14}"
)
print("-" * 70)

current_avg = threshold_analysis.groupby('segment')['win_rate'].mean()

for seg, (peak_band, cap) in thresholds.items():
    nb_avg = merged[
        (merged['segment'] == seg) &
        (merged['deal_type'] == 'New Business')
    ]['discount_pct'].mean()
    print(
        f"{seg:<15} {peak_band:>20} {cap:>18} {nb_avg:>13.1%}"
    )

## Discount Matrix — Recommended Caps by Segment and Deal Type

In [ ]:
matrix_data = con.execute("""
    SELECT
        d.segment,
        d.deal_type,
        ROUND(AVG(d.discount_pct), 4) AS avg_discount,
        ROUND(AVG(o.win_flag), 4) AS win_rate,
        COUNT(*) AS deals
    FROM deals d
    JOIN outcomes o ON d.deal_id = o.deal_id
    GROUP BY d.segment, d.deal_type
    ORDER BY d.segment, d.deal_type
""").df()

# Recommended caps based on optimal threshold analysis
caps = {
    ('SMB',        'New Business'): 0.15,
    ('SMB',        'Expansion'):    0.10,
    ('SMB',        'Renewal'):      0.08,
    ('Mid-Market', 'New Business'): 0.20,
    ('Mid-Market', 'Expansion'):    0.15,
    ('Mid-Market', 'Renewal'):      0.10,
    ('Enterprise', 'New Business'): 0.10,
    ('Enterprise', 'Expansion'):    0.08,
    ('Enterprise', 'Renewal'):      0.05,
}

matrix_data['recommended_cap'] = matrix_data.apply(
    lambda r: caps.get((r['segment'], r['deal_type']), 0.15), axis=1
)

# Pivot for heatmap
pivot = matrix_data.pivot(
    index='segment', columns='deal_type', values='recommended_cap'
)
pivot = pivot.reindex(
    index=['SMB', 'Mid-Market', 'Enterprise'],
    columns=['New Business', 'Expansion', 'Renewal']
)

print("Recommended Discount Caps:")
print()
print(
    f"{'Segment':<15} {'New Business':>14} {'Expansion':>12} {'Renewal':>10}"
)
print("-" * 53)
for seg in ['SMB', 'Mid-Market', 'Enterprise']:
    row = pivot.loc[seg]
    print(
        f"{seg:<15} {row['New Business']:>13.0%} "
        f"{row['Expansion']:>11.0%} {row['Renewal']:>9.0%}"
    )

In [ ]:
segments = ['Enterprise', 'Mid-Market', 'SMB']
deal_types = ['New Business', 'Expansion', 'Renewal']

z_values = [
    [pivot.loc[seg, dt] for dt in deal_types]
    for seg in segments
]

text_values = [
    [f"{pivot.loc[seg, dt]:.0%}" for dt in deal_types]
    for seg in segments
]

fig = go.Figure(go.Heatmap(
    z=z_values,
    x=deal_types,
    y=segments,
    text=text_values,
    texttemplate='%{text}',
    colorscale='Blues',
    showscale=False
))

fig.update_layout(
    title='Recommended Discount Caps by Segment and Deal Type',
    height=350,
    font=dict(size=13, color='#212121'),
    template='plotly_white'
)

fig.show()
fig.write_image('../outputs/10_discount_matrix.png')

## Recommendation 2: Approval SLAs

In [ ]:
sla_data = con.execute("""
    SELECT
        d.segment,
        COUNT(*) AS total_approvals,
        ROUND(AVG(a.cycle_days), 1) AS avg_days,
        PERCENTILE_CONT(0.5) WITHIN GROUP (
            ORDER BY a.cycle_days
        ) AS median_days,
        PERCENTILE_CONT(0.9) WITHIN GROUP (
            ORDER BY a.cycle_days
        ) AS p90_days
    FROM approvals a
    JOIN deals d ON a.deal_id = d.deal_id
    WHERE a.cycle_days IS NOT NULL
    GROUP BY d.segment
    ORDER BY avg_days DESC
""").df()

recommended_sla = {
    'Enterprise': 10,
    'Mid-Market': 5,
    'SMB': 2
}

print(
    f"{'Segment':<15} {'Avg Days':>10} {'Median':>10} "
    f"{'P90':>8} {'Recommended SLA':>18}"
)
print("-" * 63)
for _, row in sla_data.iterrows():
    sla = recommended_sla[row['segment']]
    print(
        f"{row['segment']:<15} {row['avg_days']:>9.1f}d "
        f"{row['median_days']:>9.1f}d {row['p90_days']:>7.1f}d "
        f"{sla:>15}d"
    )

## Recommendation 3: Rep Coaching Priorities

In [ ]:
rep_coaching = con.execute("""
    SELECT
        d.rep_id,
        COUNT(*) AS total_deals,
        ROUND(AVG(d.discount_pct), 4) AS avg_discount,
        ROUND(AVG(o.win_flag), 4) AS win_rate,
        ROUND(SUM(d.list_price * d.discount_pct), 0) AS discount_given,
        ROUND(SUM(o.final_arr), 0) AS final_arr
    FROM deals d
    JOIN outcomes o ON d.deal_id = o.deal_id
    WHERE d.deal_type = 'New Business'
    GROUP BY d.rep_id
    ORDER BY avg_discount DESC
""").df()

rep_coaching['efficiency'] = (
    rep_coaching['final_arr'] / rep_coaching['discount_given']
)

In [ ]:
efficiency_threshold = rep_coaching['efficiency'].quantile(0.25)
rep_coaching_sorted = rep_coaching.sort_values('efficiency', ascending=True)

print(f"Efficiency threshold (bottom 25%): {efficiency_threshold:.1f}x")
print()
print(
    f"{'Rep':<12} {'Deals':>7} {'Avg Disc':>10} {'Win Rate':>10} "
    f"{'Disc Given':>13} {'Efficiency':>12} {'Flags':>10}"
)
print("-" * 76)
for _, row in rep_coaching_sorted.iterrows():
    flags = []
    if row['efficiency'] <= efficiency_threshold:
        flags.append('low eff')
    if row['avg_discount'] > 0.15:
        flags.append('high disc')
    flag_str = ', '.join(flags)
    print(
        f"{row['rep_id']:<12} {row['total_deals']:>7,} "
        f"{row['avg_discount']:>9.1%} {row['win_rate']:>9.1%} "
        f"${row['discount_given']:>11,.0f} "
        f"{row['efficiency']:>11.1f}x  {flag_str}"
    )

## Recommendation 4: Pre-Approval for Expansions and Renewals

In [ ]:
pre_approval = con.execute("""
    SELECT
        d.deal_type,
        COUNT(*) AS total_deals,
        SUM(CASE WHEN a.deal_id IS NOT NULL THEN 1 ELSE 0 END) AS went_to_approval,
        ROUND(
            SUM(CASE WHEN a.deal_id IS NOT NULL THEN 1 ELSE 0 END) * 100.0
            / COUNT(*), 1
        ) AS pct_requiring_approval,
        ROUND(AVG(o.win_flag), 4) AS win_rate,
        ROUND(AVG(d.discount_pct), 4) AS avg_discount
    FROM deals d
    LEFT JOIN approvals a ON d.deal_id = a.deal_id
    JOIN outcomes o ON d.deal_id = o.deal_id
    GROUP BY d.deal_type
    ORDER BY pct_requiring_approval DESC
""").df()

print(
    f"{'Deal Type':<16} {'Deals':>8} {'To Approval':>13} "
    f"{'% Approval':>12} {'Win Rate':>10} {'Avg Disc':>10}"
)
print("-" * 71)
for _, row in pre_approval.iterrows():
    print(
        f"{row['deal_type']:<16} {row['total_deals']:>8,} "
        f"{row['went_to_approval']:>13,} "
        f"{row['pct_requiring_approval']:>11.1f}% "
        f"{row['win_rate']:>9.1%} {row['avg_discount']:>9.1%}"
    )

## Recommendation 5: Tiered SMB Approval Thresholds

In [ ]:
tiered_thresholds = {
    'SMB < $10K ARR':     {'cap': '20%', 'approver': 'None',       'sla': 'N/A'},
    'SMB $10K-$30K ARR':  {'cap': '15%', 'approver': 'Manager',    'sla': '2 days'},
    'Mid-Market':         {'cap': '20%', 'approver': 'VP Sales',   'sla': '5 days'},
    'Enterprise < 15%':   {'cap': '15%', 'approver': 'None',       'sla': 'N/A'},
    'Enterprise 15-20%':  {'cap': '20%', 'approver': 'VP Sales',   'sla': '5 days'},
    'Enterprise 20-25%':  {'cap': '25%', 'approver': 'CRO',        'sla': '5 days'},
    'Enterprise > 25%':   {'cap': 'N/A', 'approver': 'CRO + CFO', 'sla': '10 days'},
}

print(f"{'Tier':<25} {'Cap':>6} {'Approver':>12} {'SLA':>10}")
print("-" * 55)
for tier, vals in tiered_thresholds.items():
    print(
        f"{tier:<25} {vals['cap']:>6} "
        f"{vals['approver']:>12} {vals['sla']:>10}"
    )

In [ ]:
tier_impact = con.execute("""
    WITH tiered AS (
        SELECT
            d.deal_id,
            d.segment,
            d.arr,
            d.discount_pct,
            CASE
                WHEN d.segment = 'SMB'        AND d.discount_pct > 0.10 THEN 1
                WHEN d.segment = 'Mid-Market' AND d.discount_pct > 0.15 THEN 1
                WHEN d.segment = 'Enterprise' AND d.discount_pct > 0.20 THEN 1
                ELSE 0
            END AS current_approval,
            CASE
                WHEN d.segment = 'SMB' AND d.arr < 10000
                     AND d.discount_pct > 0.20 THEN 1
                WHEN d.segment = 'SMB' AND d.arr >= 10000
                     AND d.discount_pct > 0.15 THEN 1
                WHEN d.segment = 'Mid-Market'
                     AND d.discount_pct > 0.20 THEN 1
                WHEN d.segment = 'Enterprise'
                     AND d.discount_pct > 0.15 THEN 1
                ELSE 0
            END AS recommended_approval
        FROM deals d
    )
    SELECT
        segment,
        COUNT(*) AS total_deals,
        SUM(current_approval) AS current_approvals,
        SUM(recommended_approval) AS recommended_approvals,
        ROUND(SUM(current_approval) * 100.0 / COUNT(*), 1) AS current_pct,
        ROUND(SUM(recommended_approval) * 100.0 / COUNT(*), 1) AS recommended_pct
    FROM tiered
    GROUP BY segment
    ORDER BY segment
""").df()

tier_impact['change'] = (
    tier_impact['recommended_approvals'] - tier_impact['current_approvals']
)

print(
    f"{'Segment':<15} {'Deals':>8} {'Curr Appr':>12} {'Rec Appr':>11} "
    f"{'Curr %':>8} {'Rec %':>8} {'Change':>10}"
)
print("-" * 74)
for _, row in tier_impact.iterrows():
    print(
        f"{row['segment']:<15} {row['total_deals']:>8,} "
        f"{row['current_approvals']:>12,} {row['recommended_approvals']:>11,} "
        f"{row['current_pct']:>7.1f}% {row['recommended_pct']:>7.1f}% "
        f"{row['change']:>+10,.0f}"
    )

total = tier_impact[['current_approvals', 'recommended_approvals', 'change']].sum()
print(
    f"\n{'Overall':<15} {tier_impact['total_deals'].sum():>8,} "
    f"{total['current_approvals']:>12,.0f} "
    f"{total['recommended_approvals']:>11,.0f} "
    f"{total['current_approvals']/tier_impact['total_deals'].sum()*100:>7.1f}% "
    f"{total['recommended_approvals']/tier_impact['total_deals'].sum()*100:>7.1f}% "
    f"{total['change']:>+10,.0f}"
)

print("\nKey observations:")
smb_curr = tier_impact[tier_impact['segment']=='SMB']['current_approvals'].values[0]
smb_rec = tier_impact[tier_impact['segment']=='SMB']['recommended_approvals'].values[0]
mm_curr = tier_impact[tier_impact['segment']=='Mid-Market']['current_approvals'].values[0]
mm_rec = tier_impact[tier_impact['segment']=='Mid-Market']['recommended_approvals'].values[0]
ent_curr = tier_impact[tier_impact['segment']=='Enterprise']['current_approvals'].values[0]
ent_rec = tier_impact[tier_impact['segment']=='Enterprise']['recommended_approvals'].values[0]

print(
    f"  SMB approvals drop by {smb_curr-smb_rec:.0f} "
    f"(-{(smb_curr-smb_rec)/smb_curr*100:.0f}%) — deal desk bandwidth freed up"
)
print(
    f"  Mid-Market approvals drop by {mm_curr-mm_rec:.0f} "
    f"(-{(mm_curr-mm_rec)/mm_curr*100:.0f}%) — threshold raised to match optimal band"
)
print(
    f"  Enterprise approvals increase by {ent_rec-ent_curr:.0f} "
    f"(+{(ent_rec-ent_curr)/ent_curr*100:.0f}%) — managed via tiered approvers, "
    f"CRO sees only ~{ent_rec-smb_rec:.0f} deals"
)
print(
    f"  Net change: {total['change']:+.0f} approvals overall "
    f"({total['change']/total['current_approvals']*100:+.0f}%)"
)

In [ ]:
approver_volume = con.execute("""
    WITH tiered AS (
        SELECT
            d.deal_id,
            d.segment,
            d.discount_pct,
            CASE
                WHEN d.segment = 'SMB' AND d.arr < 10000
                     AND d.discount_pct > 0.20 THEN 'Manager'
                WHEN d.segment = 'SMB' AND d.arr >= 10000
                     AND d.discount_pct > 0.15 THEN 'Manager'
                WHEN d.segment = 'Mid-Market'
                     AND d.discount_pct > 0.20 THEN 'VP Sales'
                WHEN d.segment = 'Enterprise'
                     AND d.discount_pct BETWEEN 0.15 AND 0.20 THEN 'VP Sales'
                WHEN d.segment = 'Enterprise'
                     AND d.discount_pct BETWEEN 0.20 AND 0.25 THEN 'CRO'
                WHEN d.segment = 'Enterprise'
                     AND d.discount_pct > 0.25 THEN 'CRO + CFO'
                ELSE 'Auto-Approve'
            END AS recommended_approver
        FROM deals d
    )
    SELECT
        recommended_approver,
        COUNT(*) AS deals,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM tiered
    GROUP BY recommended_approver
    ORDER BY deals DESC
""").df()

print(f"{'Approver':<15} {'Deals':>8} {'Pct':>8}")
print("-" * 33)
for _, row in approver_volume.iterrows():
    print(
        f"{row['recommended_approver']:<15} {row['deals']:>8,} "
        f"{row['pct']:>7.1f}%"
    )

## Executive Memo

**Summary**

Analysis of 2,000 deals across 20 reps identifies four actions to protect 
margin, accelerate approvals, and improve new business win rates.

---

**Finding 1: Discounting beyond optimal thresholds does not improve win rates**

Win rate peaks at 10-15% for SMB, 15-20% for Mid-Market, and 5-10% for 
Enterprise. Beyond these bands, win rates decline. Enterprise reps currently 
average 16.1% discount on new business — well above the 10% optimal cap. 
$37.1M in discounts were given across the portfolio; 54% was on deals that 
were lost anyway.

*Recommendation:* Enforce segment-specific discount caps via Salesforce CPQ. 
Require VP Sales approval above cap, CRO approval above cap+5%.

---

**Finding 2: Slow approvals cost deals**

New Business deals approved within 2 days win at 54.8% vs 40.8% for deals 
taking 3-5 days — a 14 percentage point gap. Enterprise approvals average 
8 days with a P90 of 12 days.

*Recommendation:* Implement approval SLAs — SMB 2 days, Mid-Market 5 days, 
Enterprise 10 days. Escalate automatically on breach.

---

**Finding 3: Expansions and renewals should be pre-approved**

21.5% of renewals and 27.8% of expansions go through full approval despite 
win rates of 71.8% and 65.3% respectively. These deals are clogging the 
approval queue and adding unnecessary cycle time.

*Recommendation:* Auto-approve expansions and renewals within the discount 
matrix caps. Reserve deal desk bandwidth for new business.

---

**Finding 4: Five reps require coaching on discount discipline**

REP_009, REP_020, REP_005, REP_013, and REP_016 are flagged for low 
discount efficiency (below 1.6x ARR/discount). REP_010 leads the team at 
5.3x efficiency and 61.4% win rate at 11.6% avg discount — a benchmark 
for new business best practice.

*Recommendation:* Pair underperforming reps with REP_010 for deal review. 
Focus coaching on qualification discipline, not just discount behaviour.

---

**Finding 5: SMB approval threshold generates unnecessary volume; Enterprise is under-scrutinised**

45.1% of SMB deals trigger deal desk review despite averaging $17K ARR.
Meanwhile only 16.8% of Enterprise deals go to approval despite averaging 
$483K ARR and discounting at 16.1% on new business.

*Recommendation:* Implement tiered thresholds with escalation by discount 
depth. Enterprise deals under 15% auto-approve; 15-20% routed to VP Sales; 
20-25% to CRO; above 25% requires CRO + CFO sign-off. Modelling shows net 
reduction of 305 approvals (-46%) portfolio-wide — SMB down 72%, Mid-Market 
down 70% — while Enterprise coverage increases 178% with approvals routed 
to the right authority level. Phased rollout over 2-3 quarters recommended.

## Key Findings

- Enterprise reps average 16.1% discount on new business — well above the recommended 10% cap
- SMB and Mid-Market are close to optimal caps; Enterprise is the highest priority to address
- $37.1M in discounts given portfolio-wide; 54% on lost deals — pure margin waste
- 45.1% of SMB deals trigger approval despite averaging $17K ARR — threshold generates unnecessary volume
- Tiered SMB thresholds recommended: auto-approve under $10K ARR up to 20%, require approval above 15% for $10K-$30K
- Pre-approval recommended for expansions and renewals — 65-72% win rates don't justify full scrutiny
- Approval SLAs: SMB 2 days, Mid-Market 5 days, Enterprise 10 days based on P90 cycle times
- REP_009 worst efficiency at 0.8x — likely a skills issue, not a discount issue
- REP_020 anomaly — 25.4% new business win rate at moderate discount, needs investigation
- REP_010 benchmark — 5.3x efficiency, 61.4% win rate, 11.6% avg discount
- A production model would incorporate CPQ fields: contract term, payment terms, ramp discounts, competitive flags
- Discount efficiency metric assumes clean CRM data on both won and lost deals